# Steam Store Game Analysis: Identifying Market Opportunities for Game Development


## Executive Summary

This analysis of ~140,000 Steam games identifies market opportunities for game development across three key dimensions: **category**, **price**, and **release timing**. Key findings:

- **Best category bets**: Deckbuilding, Base Building, and Roguelike hybrids show high success scores with low competition. Avoid pure Action/Adventure/Indie/Single-player unless strongly differentiated.
- **Optimal pricing**: Premium ($30+) and mid-tier ($15-30) games achieve significantly higher review scores. The <$5 budget space is saturated and performs poorly.
- **Release timing**: November-December offer the best visibility. February and summer months are the most competitive with the lowest average performance.
- **Market gaps**: Cinematic + Action Roguelike, Cinematic + Base Building, and Deckbuilding + City Builder combinations have high potential with very few competitors.
- **Key success factor**: Games with demos available show a +0.40 correlation with success — the strongest single factor identified.

> **Original brief**: The market research team has asked us to analyse the current game industry to determine what category/style of game should be made, what price it should be sold at, and the best release date for optimal post-launch sales.

### Data Sources

Data harvested from SteamSpy, Steam Store API, and Steam Community API. The dataset includes game metadata, reviews, genres, community tags, categories, and estimated sales figures.


---
## Phase 1: Data Preparation

### Load, Clean, and Validate the Dataset


In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from os import path
from PIL import Image
from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator
import re
import csv
%matplotlib inline 
import json
import matplotlib_inline
import matplotlib.pyplot as plt
from matplotlib_venn import venn2, venn2_circles
import plotly.express as px


ModuleNotFoundError: No module named 'wordcloud'

Begin by loading the main games dataset to understand its structure and content.


In [ ]:
df_games = pd.read_csv('games.csv', encoding='utf-8', low_memory=False)


In [ ]:
df_games.head(5)

Examine column data types and memory usage to identify parsing issues.


In [ ]:
df_games.info()

Count missing values per column to assess data quality.


In [ ]:
df_games.isnull().sum()

The dataset contains significant missing values and malformed columns. The `price_overview` field is a JSON fragment parsed across multiple columns, and several unnamed columns carry no useful data. A systematic cleaning process is required.


In [ ]:
# Comprehensive Steam Games Data Cleaning Script
import numpy as np
import re

# Load the CSV file
df = pd.read_csv('games.csv', encoding='utf-8', low_memory=False)

# Function to combine price information from multiple columns
def combine_price_info(row):
    price_info = {}
    
    # Extract final price from price_overview
    if pd.notna(row['price_overview']):
        match = re.search(r'final\\\": (\d+)', str(row['price_overview']))
        if match:
            price_info['final'] = int(match.group(1))
    
    # Extract initial price from Initial_price
    if pd.notna(row['Initial_price']):
        match = re.search(r'initial\\\": (\d+)', str(row['Initial_price']))
        if match:
            price_info['initial'] = int(match.group(1))
    
    # Extract currency from currency
    if pd.notna(row['currency']):
        match = re.search(r'currency\\\": \\\"(\w+)\\\"', str(row['currency']))
        if match:
            price_info['currency'] = match.group(1)
    
    # Extract discount_percent from discount_percent
    if pd.notna(row['discount_percent']):
        match = re.search(r'discount_percent\\\": (\d+)', str(row['discount_percent']))
        if match:
            price_info['discount_percent'] = int(match.group(1))
    
    return price_info if price_info else np.nan

# Apply the function to create a new column with the combined price information
df['price_info'] = df.apply(combine_price_info, axis=1)

# Extract individual price components
df['final_price'] = df['price_info'].apply(lambda x: x.get('final') if isinstance(x, dict) else np.nan)
df['initial_price'] = df['price_info'].apply(lambda x: x.get('initial') if isinstance(x, dict) else np.nan)
df['currency_clean'] = df['price_info'].apply(lambda x: x.get('currency') if isinstance(x, dict) else np.nan)
df['discount_percent_clean'] = df['price_info'].apply(lambda x: x.get('discount_percent') if isinstance(x, dict) else np.nan)

# Convert release_date to datetime
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce', dayfirst=True)

# Convert is_free to boolean
df['is_free'] = df['is_free'].astype(bool)

# Create a clean dataframe with only the columns we need
clean_columns = ['app_id', 'name', 'release_date', 'is_free', 'final_price', 'initial_price', 
                'currency_clean', 'discount_percent_clean', 'languages', 'type']
df_clean = df[clean_columns].copy()

# Rename columns for clarity
df_clean.rename(columns={'currency_clean': 'currency', 'discount_percent_clean': 'discount_percent'}, inplace=True)

# Save the cleaned data to a new CSV file
df_clean.to_csv('games_cleaned.csv', index=False)

print("Data cleaning completed and saved to 'games_cleaned.csv'")
print(f"Original shape: {df.shape}, Cleaned shape: {df_clean.shape}")
print("\
Missing values in cleaned data:")
print(df_clean.isnull().sum())
print("\
Sample of cleaned data:")
df_clean.head()

In [ ]:
# we want to remove the <strong> and other weird things they have in the languages column. 

In [ ]:
df_clean.to_csv('df_games_clean.csv', index=False)

Now load the remaining datasets — reviews, categories, genres, and tags — and inspect their structure before cleaning.


In [ ]:
df_cat = pd.read_csv('categories.csv', low_memory=False)
df_genres = pd.read_csv('genres.csv', low_memory=False)
df_tags = pd.read_csv('tags.csv', low_memory=False)
df_reviews = pd.read_csv('reviews.csv', low_memory=False)


In [ ]:
df_reviews.info()

Several columns have incorrect data types (stored as `object` instead of numeric). These need conversion for proper analysis. The reviews dataset also contains text values that leaked into numeric columns during scraping.

### Cleaning the Reviews Data


In [ ]:
df_reviews.head(10)

On closer inspection, the reviews dataset contains text entries in rows 722-723 where scraped commentary has been incorrectly placed into numeric columns. These rows, along with unnamed columns and SteamSpy-specific columns, need to be removed.


In [ ]:
df_reviews.head(725)

Rows 722-723 contain text values that have entered numeric columns such as `positive`, `negative`, and `review_score`. These appear to be scraped commentary that was incorrectly parsed. We will:
1. Remove these malformed rows
2. Drop unnamed columns with no analytical value
3. Remove SteamSpy columns (not required for this analysis)


```# Remove the extra unwanted columns```
```df_reviews.drop(['steamspy_user_score', 'steamspy_score_rank','Unnamed: 12', 'Unnamed: 13', 
                 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 
                 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 
                'Unnamed: 22'], axis=1, inplace=True)```

In [ ]:
df_reviews_cleaned = pd.read_csv('reviews_cleaned_2.csv', low_memory=False)  # Suppress DtypeWarning
df_reviews_spydrop = df_reviews_cleaned.drop(
    ['steamspy_negative','steamspy_positive','steamspy_user_score', 'steamspy_score_rank', 'Unnamed: 12', 'Unnamed: 13',
     'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 
     'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22'], axis=1)  # Remove inplace=True

df_reviews_spydrop.head()

# Check if any value in 'app_id' contains letters
contains_letters = df_reviews_spydrop['app_id'].astype(str).str.contains(r'[a-zA-Z]').any()

# Print result
if contains_letters:
    print("There are values in the 'app_id' column that contain letters.")
else:
    print("All values in the 'app_id' column are numeric.")
    

    

In [ ]:
df_reviews_spydrop.info()

In [ ]:
#we need to change the datatype of most of these columns as most are integers or strings

# Convert `app_id` to int (after checking for NaNs)
df_reviews_spydrop['app_id'] = df_reviews_spydrop['app_id'].fillna(0).astype(int)

# Convert review-related columns to numeric, handling errors
cols_to_convert = ['review_score', 'positive', 'negative', 'total', 'metacritic_score', 'recommendations']
for col in cols_to_convert:
    df_reviews_spydrop[col] = pd.to_numeric(df_reviews_spydrop[col], errors='coerce')

# Display updated types
print(df_reviews_spydrop.info())


In [ ]:
df_reviews_spydrop.head(725)  

In [ ]:
df_reviews_spydrop['app_id'].count()

The numeric conversion appears successful. Let's verify by checking for any remaining text values in the numeric columns.


In [ ]:

# Check unique values in review_score_description
print("Unique values in review_score_description:")
print(df_reviews_spydrop["review_score_description"].unique())

#Check for non-numeric values in numeric columns (excluding 'review_score_description')
def is_non_numeric(value):
    if pd.isna(value) or isinstance(value, (int, float)):  # Allow NaN and numeric types
        return False
    return not str(value).replace('.', '', 1).isdigit()  # Check if non-numeric

# Apply function and sum occurrences
non_numeric = df_reviews_spydrop.drop(columns=["review_score_description"]).map(is_non_numeric).sum()
print("\nNon-numeric values in numeric columns:\n", non_numeric)

#Check data types
print("\nData types of cleaned DataFrame:")
print(df_reviews_spydrop.dtypes)

Now check the remaining datasets (categories, genres, tags) for any cleaning requirements.


In [ ]:
#df_cat = pd.read_csv('categories.csv')
#df_genres = pd.read_csv('genres.csv')
#df_tags = pd.read_csv('tags.csv')

In [ ]:
df_cat.head(30)

In [ ]:
# Most common Steam categories
cat_counts = df_cat['category'].value_counts().head(15)
print('Top 15 most common Steam categories:')
print(cat_counts.to_string())


In [ ]:
df_genres.head(20)

In [ ]:
df_genres.sample(10)

In [ ]:
df_tags.head(10)

In [ ]:
df_tags.sample(10)

All datasets are now cleaned and ready for analysis. The original raw files remain untouched — cleaned versions have been exported separately.


Export all cleaned datasets to preserve the cleaned state while keeping original files intact for reproducibility.


With data preparation complete, begin exploratory analysis to understand distributions, identify popular and unpopular games, and examine relationships between review volume, rating scores, and categories.

### Exploratory Questions:
- Which games have the most positive/negative reviews?
- How does review volume relate to rating score?
- What are the most common categories and genres?
- Which games are the most controversial (closest to 50/50 split)?


In [ ]:
df_games_cleaned =  pd.read_csv('df_games_clean.csv')
df_games_cleaned.head(5)

In [ ]:
df_reviews_cleaned = pd.read_csv('reviews_cleaned.csv')
df_reviews_cleaned.head(5)


## reviews_cleaned is incorrect 

In [ ]:
df_combined_games_reviews = pd.merge(df_games_cleaned,df_reviews_spydrop, on='app_id',how = 'inner')
df_combined_games_reviews.head(10)


In [ ]:
df_combined_games_reviews.describe()

In [ ]:
# Plot countplot 
# Distribution of review scores

plt.figure(figsize=(10, 6))
sns.countplot(data=df_combined_games_reviews.dropna(subset=['review_score']), x='review_score', order=sorted(df_combined_games_reviews['review_score'].dropna().unique()))
plt.title('Distribution of Review Scores')
plt.xlabel('Review Score')
plt.ylabel('Count')
plt.xticks(rotation=0)  # Ensure x-axis labels are readable
plt.show()


In [ ]:
# Ensure 'positive' column is numeric
df_combined_games_reviews['positive'] = pd.to_numeric(df_combined_games_reviews['positive'], errors='coerce')

# Get the top 10 games with the most positive reviews
top_10_positive_reviews = df_combined_games_reviews[['name', 'positive']].sort_values(by='positive', ascending=False).head(10)

# Plot with value labels and gradient colors
fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.Greens(np.linspace(0.3, 0.9, 10))
bars = ax.barh(range(len(top_10_positive_reviews)), top_10_positive_reviews['positive'], color=colors)
ax.set_yticks(range(len(top_10_positive_reviews)))
ax.set_yticklabels(top_10_positive_reviews['name'])
ax.set_xlabel('Number of Positive Reviews', fontsize=12, fontweight='bold')
ax.set_title('Top 10 Games with the Most Positive Reviews', fontsize=14, fontweight='bold', pad=15)
ax.invert_yaxis()
for bar, val in zip(bars, top_10_positive_reviews['positive']):
    ax.text(bar.get_width() + bar.get_width() * 0.02, bar.get_y() + bar.get_height() / 2,
            f'{int(val):,}', va='center', fontsize=9, fontweight='bold')
sns.despine()
plt.tight_layout()
plt.show()


In [ ]:
# Ensure 'negative' column is numeric
df_combined_games_reviews['negative'] = pd.to_numeric(df_combined_games_reviews['negative'], errors='coerce')

# Get the top 10 games with the most negative reviews
top_10_negative_reviews = df_combined_games_reviews[['name', 'negative']].sort_values(by='negative', ascending=False).head(10)

# Plot with value labels and gradient colors
fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.Reds(np.linspace(0.3, 0.9, 10))
bars = ax.barh(range(len(top_10_negative_reviews)), top_10_negative_reviews['negative'], color=colors)
ax.set_yticks(range(len(top_10_negative_reviews)))
ax.set_yticklabels(top_10_negative_reviews['name'])
ax.set_xlabel('Number of Negative Reviews', fontsize=12, fontweight='bold')
ax.set_title('Top 10 Games with the Most Negative Reviews', fontsize=14, fontweight='bold', pad=15)
ax.invert_yaxis()
for bar, val in zip(bars, top_10_negative_reviews['negative']):
    ax.text(bar.get_width() + bar.get_width() * 0.02, bar.get_y() + bar.get_height() / 2,
            f'{int(val):,}', va='center', fontsize=9, fontweight='bold')
sns.despine()
plt.tight_layout()
plt.show()


In [ ]:
# Top 15 games with Overwhelmingly Positive reviews
# (Using raw count rather than percentage to show the most impactful games)
plt.rcParams['font.family'] = 'DejaVu Sans'
top_overwhelmingly_pos = df_combined_games_reviews[
    df_combined_games_reviews['review_score_description'] == 'Overwhelmingly Positive'
][['name', 'positive']].sort_values(by='positive', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0.2, 0.9, 15))
bars = ax.barh(range(len(top_overwhelmingly_pos)), top_overwhelmingly_pos['positive'], color=colors)
ax.set_yticks(range(len(top_overwhelmingly_pos)))
ax.set_yticklabels(top_overwhelmingly_pos['name'])
ax.set_xlabel('Number of Positive Reviews', fontsize=12, fontweight='bold')
ax.set_title('Top 15 Overwhelmingly Positive Games on Steam', fontsize=14, fontweight='bold', pad=15)
ax.invert_yaxis()
for bar, val in zip(bars, top_overwhelmingly_pos['positive']):
    ax.text(bar.get_width() + bar.get_width() * 0.02, bar.get_y() + bar.get_height() / 2,
            f'{int(val):,}', va='center', fontsize=9, fontweight='bold')
sns.despine()
plt.tight_layout()
plt.show()


In [ ]:

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['PingFang SC', 'AppleGothic', 'Hiragino Sans GB', 'DejaVu Sans']
# Controversial games = those with positive ratio closest to 50%
df_combined_games_reviews['positive'] = pd.to_numeric(df_combined_games_reviews['positive'], errors='coerce')
df_combined_games_reviews['negative'] = pd.to_numeric(df_combined_games_reviews['negative'], errors='coerce')
df_combined_games_reviews['total_reviews'] = df_combined_games_reviews['positive'] + df_combined_games_reviews['negative']

# Filter for games with meaningful review volume
df_votes = df_combined_games_reviews[df_combined_games_reviews['total_reviews'] >= 500].copy()
df_votes['positive_ratio'] = df_votes['positive'] / df_votes['total_reviews']
df_votes['controversy_score'] = 1 - abs(df_votes['positive_ratio'] - 0.5) * 2

df_controversial = df_votes.nlargest(15, 'controversy_score')

fig, ax = plt.subplots(figsize=(12, 7))
y_pos = range(len(df_controversial))
ax.barh(y_pos, df_controversial['positive'], label='Positive Reviews', color='green', alpha=0.7)
ax.barh(y_pos, df_controversial['negative'], left=df_controversial['positive'],
        label='Negative Reviews', color='red', alpha=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(df_controversial['name'], fontsize=9)
ax.set_xlabel('Number of Reviews', fontweight='bold')
ax.set_title('Most Controversial Games (Closest to 50/50 Split)', fontweight='bold', pad=15)
ax.legend()

for i, (_, row) in enumerate(df_controversial.iterrows()):
    pct = row['positive_ratio'] * 100
    ax.text(row['total_reviews'] + row['total_reviews'] * 0.02, i,
            f'{pct:.0f}% positive', va='center', fontsize=8, fontweight='bold')

import warnings
warnings.filterwarnings('ignore', category=UserWarning)
plt.tight_layout()
plt.show()

print('MOST CONTROVERSIAL GAMES (split closest to 50/50, min 500 reviews):')
for i, (_, row) in enumerate(df_controversial.head(10).iterrows()):
    print(f'  {i+1}. {row["name"]:40s} {row["positive_ratio"]:.0%} positive  ({int(row["positive"]):,} pos / {int(row["negative"]):,} neg)')


In [ ]:
# Ensure 'recommendations' column is numeric
df_combined_games_reviews['recommendations'] = pd.to_numeric(df_combined_games_reviews['recommendations'], errors='coerce')

# Get the top 10 recommended games
top_10_recommended = df_combined_games_reviews[['name', 'recommendations']].sort_values(by='recommendations', ascending=False).head(10)

# Plot the bar chart with value labels
fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.Blues(np.linspace(0.4, 0.9, 10))
bars = ax.barh(range(len(top_10_recommended)), top_10_recommended['recommendations'], color=colors)
ax.set_yticks(range(len(top_10_recommended)))
ax.set_yticklabels(top_10_recommended['name'])
ax.set_xlabel('Number of Recommendations', fontsize=12, fontweight='bold')
ax.set_title('Top 10 Recommended Games on Steam', fontsize=14, fontweight='bold', pad=15)
ax.invert_yaxis()

# Add formatted value labels at the end of each bar
for bar, val in zip(bars, top_10_recommended['recommendations']):
    ax.text(bar.get_width() + bar.get_width() * 0.02, bar.get_y() + bar.get_height() / 2,
            f'{int(val):,}', va='center', fontsize=9, fontweight='bold')

sns.despine()
plt.tight_layout()
plt.show()


This list shows the top 10 games by raw recommendation count — a metric that heavily favors games with large player bases rather than necessarily indicating the highest quality games. A game with millions of players will naturally accumulate more recommendations than a niche gem with a passionate but smaller audience. For a more balanced view, consider recommendation rate (recommendations per owner) alongside absolute counts.


In [ ]:
# Compare top positive vs top recommended — which games excel at both?

top_10_positive = df_combined_games_reviews[['name', 'positive']].sort_values(by='positive', ascending=False).head(10)
top_10_recommended = df_combined_games_reviews[['name', 'recommendations']].sort_values(by='recommendations', ascending=False).head(10)

# Merge to find games appearing in BOTH lists (inner) and mark which list each came from
top_10_positive['in_top_positive'] = True
top_10_recommended['in_top_recommended'] = True

combined = pd.merge(top_10_positive[['name', 'positive', 'in_top_positive']],
                    top_10_recommended[['name', 'recommendations', 'in_top_recommended']],
                    on='name', how='outer').fillna(False)

# Show overlap
overlap = combined[combined['in_top_positive'] & combined['in_top_recommended']]
only_positive = combined[combined['in_top_positive'] & ~combined['in_top_recommended']]
only_recommended = combined[~combined['in_top_positive'] & combined['in_top_recommended']]

print('GAMES IN BOTH LISTS (Overlap):')
if len(overlap) > 0:
    for _, row in overlap.iterrows():
        print(f'  {row["name"]:40s} {int(row["positive"]):>8,} positive  {int(row["recommendations"]):>8,} recommendations')
else:
    print('  (None — the top-10 positive and top-10 recommended lists share no games)')
print()
print(f'Only in top-10 positive: {len(only_positive)} games')
print(f'Only in top-10 recommended: {len(only_recommended)} games')
print(f'Overlap: {len(overlap)} games')
print()

# Grouped bar chart: side-by-side comparison of the union of both lists
plot_df = combined.sort_values('positive', ascending=False).copy()
# Normalize for side-by-side display
pos_max = plot_df['positive'].max()
rec_max = plot_df['recommendations'].max()
plot_df['positive_norm'] = plot_df['positive'] / pos_max * 100
plot_df['recommendations_norm'] = plot_df['recommendations'] / rec_max * 100

# Better visualization: show two separate bars for each game (positive count, recommendation count)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left bar chart: Positive reviews
top_10_positive_sorted = top_10_positive.sort_values('positive', ascending=True)
bars1 = ax1.barh(range(len(top_10_positive_sorted)), top_10_positive_sorted['positive'],
                 color='green', alpha=0.7, edgecolor='black', linewidth=0.5)
ax1.set_yticks(range(len(top_10_positive_sorted)))
ax1.set_yticklabels(top_10_positive_sorted['name'], fontsize=9)
ax1.set_title('Top 10 by Positive Reviews', fontweight='bold')
ax1.set_xlabel('Positive Reviews', fontweight='bold')
for bar, val in zip(bars1, top_10_positive_sorted['positive']):
    ax1.text(bar.get_width() + bar.get_width() * 0.01, bar.get_y() + bar.get_height() / 2,
            f'{int(val):,}', va='center', fontsize=8)

# Right bar chart: Recommendations
top_10_rec_sorted = top_10_recommended.sort_values('recommendations', ascending=True)
bars2 = ax2.barh(range(len(top_10_rec_sorted)), top_10_rec_sorted['recommendations'],
                 color='steelblue', alpha=0.7, edgecolor='black', linewidth=0.5)
ax2.set_yticks(range(len(top_10_rec_sorted)))
ax2.set_yticklabels(top_10_rec_sorted['name'], fontsize=9)
ax2.set_title('Top 10 by Recommendations', fontweight='bold')
ax2.set_xlabel('Recommendations', fontweight='bold')
for bar, val in zip(bars2, top_10_rec_sorted['recommendations']):
    ax2.text(bar.get_width() + bar.get_width() * 0.01, bar.get_y() + bar.get_height() / 2,
            f'{int(val):,}', va='center', fontsize=8)

for ax in [ax1, ax2]:
    sns.despine(ax=ax)

plt.suptitle("Comparing Steam's Most Popular Games: Positive Reviews vs Recommendations",
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Preliminary: Game releases over time
# Using the games+reviews merged dataset (one row per game, ~140K games)
# Phase 2 (below) will merge in genres, categories, and tags for deeper analysis.


In [ ]:
# Ensure 'release_date' is in datetime format
df_games_timeline = df_combined_games_reviews.copy()
df_games_timeline['release_date'] = pd.to_datetime(df_games_timeline['release_date'], errors='coerce')

# Extract the year and count unique games per year
df_games_timeline['release_year'] = df_games_timeline['release_date'].dt.year
yearly_releases = df_games_timeline['release_year'].value_counts().sort_index().reset_index()
yearly_releases.columns = ['year', 'count']


In [ ]:
yearly_releases_filtered = yearly_releases[yearly_releases['year'] >= 2005]

plt.figure(figsize=(12,6))
sns.barplot(data=yearly_releases_filtered, x='year', y='count', color='royalblue')
plt.title('Number of New Games Released Each Year', fontsize=14, fontweight='bold')
plt.xlabel('Year', fontsize=12)
plt.ylabel('Number of Games Released', fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
sns.despine()
plt.tight_layout()
plt.show()


As expected, the number of games released on Steam has grown exponentially over the years, driven by platform growth, accessibility of development tools, and the rise of indie publishing.


In [ ]:
# Compute cumulative sum of games released over time
yearly_releases_sorted = yearly_releases.sort_values('year')
yearly_releases_sorted['total_games'] = yearly_releases_sorted['count'].cumsum()

plt.figure(figsize=(12,6))
sns.lineplot(data=yearly_releases_sorted, x='year', y='total_games', marker='o', linewidth=2, color='darkred')
plt.title('Total Number of Games Available on Steam Over Time', fontsize=14, fontweight='bold')
plt.xlabel('Year', fontsize=12)
plt.ylabel('Total Number of Games', fontsize=12)
plt.grid(True, alpha=0.3)
sns.despine()
plt.tight_layout()
plt.show()


## Phase 2: Merge Genres, Categories & Tags
Now we enrich the combined games+reviews dataframe with genre, category, and tag data to answer our business questions.

In [ ]:
import numpy as np

# Load genre, category, and tag data
df_genres = pd.read_csv('genres.csv')
df_categories = pd.read_csv('categories.csv')
df_tags = pd.read_csv('tags.csv')

print(f'Genres: {len(df_genres):,} rows, {df_genres["genre"].nunique()} unique')
print(f'Categories: {len(df_categories):,} rows, {df_categories["category"].nunique()} unique')
print(f'Tags: {len(df_tags):,} rows, {df_tags["tag"].nunique()} unique')

# --- One-hot encode top genres ---
top_genres = df_genres['genre'].value_counts().head(15).index
genre_pivot = df_genres[df_genres['genre'].isin(top_genres)].pivot_table(
    index='app_id', columns='genre', aggfunc='size', fill_value=0
)
genre_pivot = (genre_pivot > 0).astype(int)
genre_pivot.columns = [f'genre_{col}' for col in genre_pivot.columns]
print(f'Genre pivot: {genre_pivot.shape[0]:,} apps, {genre_pivot.shape[1]} columns')

# --- One-hot encode top categories ---
top_cats = df_categories['category'].value_counts().head(15).index
cat_pivot = df_categories[df_categories['category'].isin(top_cats)].pivot_table(
    index='app_id', columns='category', aggfunc='size', fill_value=0
)
cat_pivot = (cat_pivot > 0).astype(int)
cat_pivot.columns = [f'cat_{col}' for col in cat_pivot.columns]
print(f'Category pivot: {cat_pivot.shape[0]:,} apps, {cat_pivot.shape[1]} columns')

# --- One-hot encode top tags ---
# Top 150 most common tags + curated niche tags
top_tags = df_tags['tag'].value_counts().head(150).index.tolist()
curated_tags = ['Farming', 'Farming Sim', 'Deckbuilding', 'Roguelike Deckbuilder', 'Souls-like', 'City Builder', 'Walking Simulator']
for t in curated_tags:
    if t not in top_tags and t in df_tags['tag'].unique():
        top_tags.append(t)
tag_pivot = df_tags[df_tags['tag'].isin(top_tags)].pivot_table(
    index='app_id', columns='tag', aggfunc='size', fill_value=0
)
tag_pivot = (tag_pivot > 0).astype(int)
tag_pivot.columns = [f'tag_{col}' for col in tag_pivot.columns]
print(f'Tag pivot: {tag_pivot.shape[0]:,} apps, {tag_pivot.shape[1]} columns')

# --- Merge into main dataframe ---
df = df_combined_games_reviews.copy()
df = df.merge(genre_pivot, on='app_id', how='left').fillna(0)
df = df.merge(cat_pivot, on='app_id', how='left').fillna(0)
df = df.merge(tag_pivot, on='app_id', how='left').fillna(0)

print(f'\nFinal merged shape: {df.shape}')
print(f'Columns: {list(df.columns[:20])}...')

## Phase 3: Feature Engineering
Create price tiers, review categories, release window features, and success metrics.

In [ ]:
# --- Price Tiers ---
def assign_price_tier(price):
    if pd.isna(price) or price == 0:
        return 'Free'
    elif price < 500:
        return 'Budget (<$5)'
    elif price < 1500:
        return 'Low ($5-15)'
    elif price < 3000:
        return 'Mid ($15-30)'
    else:
        return 'Premium ($30+)'

df['price_tier'] = df['final_price'].apply(assign_price_tier)

# --- Review Category (simplified) ---
def simplify_review(desc):
    if pd.isna(desc):
        return 'Unknown'
    if desc in ['Overwhelmingly Positive', 'Very Positive', 'Mostly Positive']:
        return 'Positive'
    elif desc in ['Mixed']:
        return 'Mixed'
    else:
        return 'Negative'

df['review_simple'] = df['review_score_description'].apply(simplify_review)

# --- Release Date Features ---
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['release_year'] = df['release_date'].dt.year
df['release_month'] = df['release_date'].dt.month
df['release_quarter'] = df['release_date'].dt.quarter

# Season mapping
season_map = {12:'Winter', 1:'Winter', 2:'Winter',
              3:'Spring', 4:'Spring', 5:'Spring',
              6:'Summer', 7:'Summer', 8:'Summer',
              9:'Autumn', 10:'Autumn', 11:'Autumn'}
df['release_season'] = df['release_month'].map(season_map)

# --- Game Age Category ---
current_year = 2024
def age_category(year):
    if pd.isna(year): return 'Unknown'
    age = current_year - year
    if age <= 1: return 'New (0-1 years)'
    elif age <= 3: return 'Recent (1-3 years)'
    elif age <= 7: return 'Established (3-7 years)'
    else: return 'Legacy (7+ years)'

df['age_category'] = df['release_year'].apply(age_category)

# --- Success Metric (Bayesian weighted score) ---
C = 100  # shrinkage factor
total_global = df['total'].sum()
global_avg_rating = (df['positive'].sum() / total_global) if total_global > 0 else 0.5

def bayesian_rating(row):
    if pd.isna(row['total']) or row['total'] == 0:
        return global_avg_rating * 100
    observed = row['positive'] / row['total']
    return (row['total'] / (row['total'] + C)) * observed + (C / (row['total'] + C)) * global_avg_rating

df['success_score'] = df.apply(bayesian_rating, axis=1) * 100

print(f'Price tier distribution:')
print(df['price_tier'].value_counts())
print(f'\nReview distribution:')
print(df['review_simple'].value_counts())
print(f'\nAge category distribution:')
print(df['age_category'].value_counts())
print(f'\nRelease years: {int(df["release_year"].min())} - {int(df["release_year"].max())}')

## Phase 4: Visualizations & Analysis
All 5 charts rebuilt to query real data from our merged dataframe.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def create_category_performance_bubble_chart():
    """
    Bubble chart: X = avg reviews (sales proxy), Y = avg review score,
    size = game count, color = avg price.
    Groups by Steam categories (from cat_* columns).
    """
    cat_cols = [c for c in df.columns if c.startswith('cat_')]
    results = []
    for col in cat_cols:
        subset = df[df[col] == 1]
        if len(subset) < 50:
            continue
        results.append({
            'category': col.replace('cat_', ''),
            'avg_total_reviews': subset['total'].mean(),
            'avg_review_score': subset['review_score'].mean(),
            'game_count': len(subset),
            'avg_price': subset['final_price'].mean()
        })
    
    df_viz = pd.DataFrame(results)
    
    fig, ax = plt.subplots(figsize=(14, 10))
    
    scatter = ax.scatter(
        df_viz['avg_total_reviews'],
        df_viz['avg_review_score'],
        s=df_viz['game_count'] / 100,
        c=df_viz['avg_price'],
        alpha=0.7, cmap='viridis', edgecolors='black', linewidth=0.5
    )
    
    for i, row in df_viz.iterrows():
        ax.annotate(row['category'],
                    (row['avg_total_reviews'], row['avg_review_score']),
                    xytext=(5, 5), textcoords='offset points', fontsize=9, ha='left')
    
    ax.set_xlabel('Average Total Reviews (Sales Proxy)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Average Review Score (Quality)', fontsize=12, fontweight='bold')
    ax.set_title('Game Category Performance vs Market Saturation\n(Bubble size = Market saturation, Color = Average price)',
                 fontsize=14, fontweight='bold', pad=20)
    
    cbar = plt.colorbar(scatter)
    cbar.set_label('Average Price ($)', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.axhline(y=6, color='red', linestyle='--', alpha=0.5)
    ax.axvline(x=3000, color='red', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()

create_category_performance_bubble_chart()

In [ ]:
def create_price_performance_heatmap():
    cat_cols = [c for c in df.columns if c.startswith('cat_')]
    results = []
    for col in cat_cols:
        subset = df[df[col] == 1]
        if len(subset) < 50:
            continue
        results.append({
            'Category': col.replace('cat_', ''),
            'Avg_Price': subset['final_price'].mean() / 100,
            'Avg_Review_Score': subset['review_score'].mean(),
            'Market_Size': len(subset),
            'Correlation': subset[['final_price', 'review_score']].dropna().corr().iloc[0, 1]
        })
    df_viz = pd.DataFrame(results)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

    scatter = ax1.scatter(
        df_viz['Avg_Price'], df_viz['Avg_Review_Score'],
        s=df_viz['Market_Size'] / 50, c=df_viz['Correlation'],
        cmap='RdYlBu', alpha=0.8, edgecolors='black', linewidth=0.5, vmin=-0.3, vmax=0.7
    )
    for _, row in df_viz.iterrows():
        ax1.annotate(row['Category'], (row['Avg_Price'], row['Avg_Review_Score']),
                     fontsize=8, alpha=0.9)
    cbar1 = plt.colorbar(scatter, ax=ax1)
    cbar1.set_label('Price-Review Correlation', fontweight='bold')
    ax1.set_xlabel('Average Price ($)', fontsize=11, fontweight='bold')
    ax1.set_ylabel('Average Review Score', fontsize=11, fontweight='bold')
    ax1.set_title('Price vs Quality by Category (Size = Market size, Color = Correlation)',
                  fontweight='bold', pad=15)
    ax1.axhline(y=df_viz['Avg_Review_Score'].mean(), color='gray', linestyle='--', alpha=0.5)
    ax1.grid(True, alpha=0.3)

    df_sorted = df_viz.sort_values('Correlation', ascending=True)
    colors = plt.cm.RdYlBu((df_sorted['Correlation'].clip(-0.3, 0.7) + 0.3) / 1.0)
    bars = ax2.barh(range(len(df_sorted)), df_sorted['Correlation'], color=colors, alpha=0.8)
    ax2.set_yticks(range(len(df_sorted)))
    ax2.set_yticklabels(df_sorted['Category'], fontsize=9)
    ax2.axvline(x=0, color='black', linewidth=0.5)
    ax2.set_xlabel('Correlation: Price vs Review Score', fontsize=11, fontweight='bold')
    ax2.set_title('Does Higher Price Mean Better Reviews? (Bar color = strength & direction)',
                  fontweight='bold', pad=15)
    for bar, val in zip(bars, df_sorted['Correlation']):
        ax2.text(val + 0.01 if val >= 0 else val - 0.06, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=8)

    plt.tight_layout()
    plt.show()

    print('KEY INSIGHT: Most categories show WEAK positive correlation (0.1-0.3)')
    print('  - Categories above 0.3: higher price genuinely signals higher quality')
    print('  - Categories near 0.0: price and quality are unrelated')
    print('  - Premium pricing works best in: ' +
          ', '.join(df_viz.nlargest(5, 'Correlation')['Category'].tolist()))
# Call the function
create_price_performance_heatmap()

In [ ]:
def create_release_timing_analysis():
    """
    Multi-panel chart: release timing effects on performance.
    """
    df_clean = df[df['release_year'].between(2015, 2024)].copy()
    
    # Monthly aggregation
    monthly = df_clean.groupby('release_month').agg(
        avg_reviews=('total', 'mean'),
        review_score=('review_score', 'mean'),
        releases=('app_id', 'count')
    ).reset_index()
    
    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                   'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    monthly['month_name'] = monthly['release_month'].apply(lambda x: month_names[int(x)-1])
    
    # Yearly aggregation
    yearly = df_clean.groupby('release_year').agg(
        avg_reviews=('total', 'mean'),
        review_score=('review_score', 'mean'),
        games_released=('app_id', 'count')
    ).reset_index()
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    x_pos = np.arange(len(monthly))
    
    # Chart 1: Monthly sales performance
    p75 = monthly['avg_reviews'].quantile(0.75)
    p25 = monthly['avg_reviews'].quantile(0.25)
    bars1 = ax1.bar(x_pos, monthly['avg_reviews'],
                    color=['red' if x < p25 else 'green' if x > p75 else 'orange'
                           for x in monthly['avg_reviews']], alpha=0.7)
    ax1.set_xlabel('Release Month')
    ax1.set_ylabel('Average Total Reviews')
    ax1.set_title('Sales Performance by Release Month\n(Green = Best, Red = Avoid)', fontweight='bold')
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(monthly['month_name'], rotation=45)
    for bar, value in zip(bars1, monthly['avg_reviews']):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                f'{value:.0f}', ha='center', va='bottom', fontweight='bold', fontsize=8)
    
    # Chart 2: Monthly quality vs competition
    ax2_twin = ax2.twinx()
    ax2.plot(x_pos, monthly['review_score'], 'bo-', linewidth=2, label='Quality Score')
    ax2_twin.plot(x_pos, monthly['releases'], 'rs-', linewidth=2, label='Competition Level')
    ax2.set_xlabel('Release Month')
    ax2.set_ylabel('Average Review Score', color='blue')
    ax2_twin.set_ylabel('Number of Releases (Competition)', color='red')
    ax2.set_title('Quality vs Competition by Month', fontweight='bold')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(monthly['month_name'], rotation=45)
    ax2.tick_params(axis='y', labelcolor='blue')
    ax2_twin.tick_params(axis='y', labelcolor='red')
    
    # Chart 3: Yearly trend
    ax3_twin = ax3.twinx()
    ax3.bar(yearly['release_year'], yearly['avg_reviews'], alpha=0.6, color='skyblue', label='Avg Reviews')
    ax3_twin.plot(yearly['release_year'], yearly['review_score'], 'ro-', linewidth=3, label='Quality Score')
    ax3.set_xlabel('Release Year')
    ax3.set_ylabel('Average Total Reviews', color='blue')
    ax3_twin.set_ylabel('Average Review Score', color='red')
    ax3.set_title('Market Saturation Impact Over Time', fontweight='bold')
    ax3.tick_params(axis='y', labelcolor='blue')
    ax3_twin.tick_params(axis='y', labelcolor='red')
    
    # Chart 4: Market saturation scatter
    sc = ax4.scatter(yearly['games_released'], yearly['avg_reviews'],
                     s=100, c=yearly['release_year'], cmap='viridis', alpha=0.7)
    z = np.polyfit(yearly['games_released'], yearly['avg_reviews'], 1)
    p = np.poly1d(z)
    ax4.plot(yearly['games_released'], p(yearly['games_released']), "r--", alpha=0.8, linewidth=2)
    ax4.set_xlabel('Total Games Released (Market Saturation)')
    ax4.set_ylabel('Average Total Reviews per Game')
    ax4.set_title('Impact of Market Saturation on Individual Game Performance', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    return monthly, yearly

df_monthly, df_yearly = create_release_timing_analysis()

In [ ]:
def create_longevity_analysis():
    """
    Analysis of game performance over time and long-term sustainability.
    """
    age_order = ['New (0-1 years)', 'Recent (1-3 years)', 'Established (3-7 years)', 'Legacy (7+ years)']
    age_data = df.groupby('age_category', observed=True).agg(
        review_score=('review_score', 'mean'),
        total_reviews=('total', 'mean'),
        game_count=('app_id', 'count'),
        avg_price=('final_price', 'mean')
    ).reindex(age_order).reset_index()
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    categories = age_data['age_category']
    x_pos = np.arange(len(categories))
    
    # Chart 1: Quality by age
    bars1 = ax1.bar(x_pos, age_data['review_score'],
                    color=['red', 'orange', 'lightgreen', 'green'], alpha=0.7)
    ax1.set_xlabel('Game Age Category')
    ax1.set_ylabel('Average Review Score')
    ax1.set_title('Game Quality Improves with Age\n(Legacy games perform best)', fontweight='bold')
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels([cat.replace(' ', '\n') for cat in categories], ha='center')
    for bar, value in zip(bars1, age_data['review_score']):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f'{value:.2f}', ha='center', va='bottom', fontweight='bold')
    
    # Chart 2: Reviews vs Price by age
    ax2_twin = ax2.twinx()
    bars2 = ax2.bar(x_pos, age_data['total_reviews'], alpha=0.6, color='skyblue', label='Total Reviews')
    ax2_twin.plot(x_pos, age_data['avg_price'], 'ro-', linewidth=3, markersize=8, label='Avg Price')
    ax2.set_xlabel('Game Age Category')
    ax2.set_ylabel('Average Total Reviews', color='blue')
    ax2_twin.set_ylabel('Average Price ($)', color='red')
    ax2.set_title('Sales Volume vs Pricing by Age', fontweight='bold')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels([cat.replace(' ', '\n') for cat in categories], ha='center')
    ax2.tick_params(axis='y', labelcolor='blue')
    ax2_twin.tick_params(axis='y', labelcolor='red')
    
    # Chart 3: Market distribution by age
    sizes = age_data['game_count']
    colors_pie = ['#ff9999', '#ffcc99', '#99ff99', '#99ccff']
    wedges, texts, autotexts = ax3.pie(sizes, explode=(0.05, 0.05, 0.05, 0.1),
                                        labels=categories, colors=colors_pie,
                                        autopct='%1.1f%%', shadow=True, startangle=90)
    ax3.set_title('Market Distribution by Game Age', fontweight='bold')
    
    # Chart 4: Sustainability score
    sustainability_score = []
    for i in range(len(age_data)):
        q_norm = age_data['review_score'].iloc[i] / age_data['review_score'].max()
        s_norm = age_data['total_reviews'].iloc[i] / age_data['total_reviews'].max()
        sustainability_score.append((q_norm + s_norm) / 2)
    
    bars4 = ax4.barh(categories, sustainability_score,
                     color=['red', 'orange', 'lightgreen', 'green'], alpha=0.7)
    ax4.set_xlabel('Sustainability Score (Quality + Sales)')
    ax4.set_title('Long-term Sustainability by Age Category', fontweight='bold')
    ax4.set_xlim(0, 1)
    for bar, value in zip(bars4, sustainability_score):
        ax4.text(value + 0.02, bar.get_y() + bar.get_height()/2,
                f'{value:.3f}', va='center', ha='left', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    return age_data

df_age = create_longevity_analysis()

In [ ]:
def create_market_opportunity_dashboard():
    genre_cols = [c for c in df.columns if c.startswith('genre_')]
    cat_cols = [c for c in df.columns if c.startswith('cat_')]
    # Exclude non-gameplay categories
    exclude_cats = {'Free To Play', 'Family Sharing', 'Steam Cloud', 'Steam Trading Cards',
                    'Remote Play Together', 'Includes Source SDK', 'Commentary available',
                    'Stats', 'Includes level editor', 'Captions available',
                    'Partial Controller Support', 'Valve Anti-Cheat', 'Steam Achievements',
                    'Steam Leaderboards', 'Steam Workshop', 'Shared/Split Screen',
                    'Cross-Platform Multiplayer', 'Remote Play on TV', 'Remote Play on Phone',
                    'Remote Play on Tablet', 'Remote Play on Phone/Tablet'}
    cat_cols = [c for c in cat_cols if c.replace('cat_', '') not in exclude_cats]

    combos = []
    for g in genre_cols[:8]:
        for c in cat_cols[:8]:
            mask = (df[g] == 1) & (df[c] == 1)
            subset = df[mask]
            if len(subset) < 30:
                continue
            combos.append({
                'combo': f'{g.replace("genre_","")}/{c.replace("cat_","")}',
                'game_count': len(subset),
                'avg_review_score': subset['review_score'].mean(),
                'avg_total_reviews': subset['total'].mean(),
                'avg_price': subset['final_price'].mean()
            })
    df_combos = pd.DataFrame(combos)

    df_combos['quality_pct'] = df_combos['avg_review_score'].rank(pct=True)
    df_combos['saturation_pct'] = df_combos['game_count'].rank(pct=True)
    df_combos['market_potential'] = (df_combos['quality_pct'] * 0.6 + (1 - df_combos['saturation_pct']) * 0.4) * 100

    opportunities = df_combos.nlargest(8, 'market_potential')
    oversaturated = df_combos.nlargest(5, 'game_count')

    fig = plt.figure(figsize=(18, 14))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

    ax1 = fig.add_subplot(gs[0, :2])
    scatter1 = ax1.scatter(
        df_combos['game_count'], df_combos['avg_review_score'],
        s=df_combos['avg_total_reviews'] / 100, c=df_combos['market_potential'],
        cmap='RdYlGn', alpha=0.6, edgecolors='gray', linewidth=0.3
    )
    for _, row in opportunities.iterrows():
        ax1.annotate(row['combo'], (row['game_count'], row['avg_review_score']),
                     fontsize=9, fontweight='bold', ha='center',
                     bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))
    for _, row in oversaturated.iterrows():
        ax1.annotate(row['combo'], (row['game_count'], row['avg_review_score']),
                     fontsize=8, ha='center',
                     bbox=dict(boxstyle='round,pad=0.2', facecolor='red', alpha=0.2))
    ax1.set_xlabel('Game Count (Saturation)', fontsize=11, fontweight='bold')
    ax1.set_ylabel('Average Review Score (Quality)', fontsize=11, fontweight='bold')
    ax1.set_title('Market Opportunity Map (Top-left = Best opportunity, Dot size = Review volume)',
                  fontweight='bold', pad=15)
    cbar1 = plt.colorbar(scatter1, ax=ax1)
    cbar1.set_label('Market Potential Score', fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.axvline(x=df_combos['game_count'].median(), color='red', linestyle='--', alpha=0.4)
    ax1.axhline(y=df_combos['avg_review_score'].median(), color='red', linestyle='--', alpha=0.4)
    ax1.text(df_combos['game_count'].max()*0.7, df_combos['avg_review_score'].max()*0.95,
             'HIGH QUALITY\nLOW COMPETITION', fontsize=10, color='green', fontweight='bold', ha='center')

    ax2 = fig.add_subplot(gs[0, 2])
    opp_sorted = opportunities.sort_values('market_potential')
    colors2 = plt.cm.RdYlGn(opp_sorted['market_potential'] / 100)
    bars2 = ax2.barh(range(len(opp_sorted)), opp_sorted['market_potential'], color=colors2, alpha=0.8)
    ax2.set_yticks(range(len(opp_sorted)))
    ax2.set_yticklabels(opp_sorted['combo'], fontsize=8)
    ax2.set_xlabel('Market Potential', fontweight='bold')
    ax2.set_title('Top Opportunities', fontweight='bold', pad=10)
    for bar, val in zip(bars2, opp_sorted['market_potential']):
        ax2.text(val + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.0f}', va='center', fontsize=8)

    ax3 = fig.add_subplot(gs[1, 0])
    oversat_sorted = oversaturated.sort_values('game_count', ascending=True)
    ax3.barh(range(len(oversat_sorted)), oversat_sorted['game_count'], color='coral', alpha=0.8)
    ax3.set_yticks(range(len(oversat_sorted)))
    ax3.set_yticklabels(oversat_sorted['combo'], fontsize=8)
    ax3.set_xlabel('Number of Games', fontweight='bold')
    ax3.set_title('Most Saturated Combos (Avoid)', fontweight='bold', pad=10)
    for i, (_, row) in enumerate(oversat_sorted.iterrows()):
        ax3.text(row['game_count'] + 50, i, f'{int(row["game_count"]):,}', va='center', fontsize=8)

    ax4 = fig.add_subplot(gs[1, 1])
    cat_pricing = df_combos.groupby('combo').agg({'avg_price': 'mean', 'avg_review_score': 'mean'}).reset_index()
    ax4.scatter(cat_pricing['avg_price'] / 100, cat_pricing['avg_review_score'],
                alpha=0.7, c='steelblue', s=60)
    for _, row in cat_pricing.iterrows():
        parts = row['combo'].split('/')
        label = parts[0] if len(parts) > 1 else parts[0]
        ax4.annotate(label, (row['avg_price'] / 100, row['avg_review_score']), fontsize=6, alpha=0.7)
    ax4.set_xlabel('Average Price ($)', fontweight='bold')
    ax4.set_ylabel('Average Review Score', fontweight='bold')
    ax4.set_title('Pricing vs Quality by Category Combo', fontweight='bold', pad=10)
    ax4.grid(True, alpha=0.3)

    ax5 = fig.add_subplot(gs[1, 2])
    top_combos = df_combos.nlargest(15, 'market_potential')
    pivot_data = top_combos[['combo', 'market_potential']].set_index('combo')
    sns.heatmap(pivot_data.T, annot=True, fmt='.0f', cmap='RdYlGn',
                cbar_kws={'label': 'Potential'}, ax=ax5, linewidths=0.5)
    ax5.set_title('Top Opportunities by Score', fontweight='bold', pad=10)
    ax5.set_xlabel('')
    ax5.set_ylabel('')

    ax6 = fig.add_subplot(gs[2, :])
    ax6.axis('off')
    strategy_lines = ['STRATEGIC RECOMMENDATIONS', '']
    strategy_lines.append('BEST OPPORTUNITIES (high quality + low competition):')
    for _, row in opportunities.head(5).iterrows():
        strategy_lines.append(f'  * {row["combo"]}: {row["avg_review_score"]:.1f} score, {int(row["game_count"]):,} games, ${row["avg_price"]/100:.2f} avg')
    strategy_lines.append('')
    strategy_lines.append('AVOID (oversaturated):')
    for _, row in oversaturated.iterrows():
        strategy_lines.append(f'  * {row["combo"]}: {int(row["game_count"]):,} games')
    strategy_lines.append('')
    strategy_lines.append('PRICING: Target $15-30 mid-tier for quality indie, $30+ premium for high production')
    strategy_text = '\n'.join(strategy_lines)

    ax6.text(0.1, 0.5, strategy_text, fontsize=11, fontfamily='monospace', va='center',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    ax6.set_title('ACTIONABLE INSIGHTS', fontweight='bold', pad=10, fontsize=13)

    fig.subplots_adjust(left=0.05, right=0.95, bottom=0.05, top=0.93, hspace=0.3, wspace=0.3)
    plt.show()

    print('SUMMARY OF KEY FINDINGS:')
    print(f'  - {len(opportunities)} strong market opportunities identified')
    print(f'  - {len(oversaturated)} areas to avoid due to saturation')
    print(f'  - Best combo: {opportunities.iloc[0]["combo"]} (potential: {opportunities.iloc[0]["market_potential"]:.0f})')
    return df_combos
# Call the function
create_market_opportunity_dashboard()

## Phase 5: Advanced Analysis
Deep dive into success factors, tag pair market gaps, standout games, and niche opportunities.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
print("=" * 60)
print("ANALYSIS 1: WHAT MAKES A GAME SUCCESSFUL?")
print("=" * 60)
feature_cols = (
    [c for c in df.columns if c.startswith('genre_')] +
    [c for c in df.columns if c.startswith('cat_')] +
    [c for c in df.columns if c.startswith('tag_')]
)
correlations = []
for col in feature_cols:
    corr = df[col].corr(df['success_score'])
    mean_val = df.loc[df[col] == 1, 'success_score'].mean()
    count = (df[col] == 1).sum()
    correlations.append({'feature': col, 'correlation': corr, 'avg_success': mean_val, 'count': count})
df_corr = pd.DataFrame(correlations).sort_values('correlation', ascending=False)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 10))
top_pos = df_corr.head(15)
top_neg = df_corr.tail(15).iloc[::-1]
ax1.barh(range(len(top_pos)), top_pos['correlation'].values, color='green', alpha=0.7)
ax1.set_yticks(range(len(top_pos)))
ax1.set_yticklabels([c.replace('tag_', '').replace('genre_', '').replace('cat_', '') for c in top_pos['feature'].values])
ax1.set_xlabel('Correlation with Success Score')
ax1.set_title('TOP 15 POSITIVE SUCCESS FACTORS', fontweight='bold')
ax1.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
ax2.barh(range(len(top_neg)), top_neg['correlation'].values, color='red', alpha=0.7)
ax2.set_yticks(range(len(top_neg)))
ax2.set_yticklabels([c.replace('tag_', '').replace('genre_', '').replace('cat_', '') for c in top_neg['feature'].values])
ax2.set_xlabel('Correlation with Success Score')
ax2.set_title('TOP 15 NEGATIVE SUCCESS FACTORS', fontweight='bold')
ax2.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()
print('\nTop features by avg success score (min 500 games):')
top_avg = df_corr[df_corr['count'] >= 500].sort_values('avg_success', ascending=False).head(10)
for _, row in top_avg.iterrows():
    fname = row['feature'].replace('tag_', '').replace('genre_', '').replace('cat_', '')
    print(f'  {fname:30s} avg_success={row["avg_success"]:6.1f}  games={int(row["count"]):,}  corr={row["correlation"]:+.3f}')
print('\nLowest features by avg success score (min 500 games):')
bot_avg = df_corr[df_corr['count'] >= 500].sort_values('avg_success', ascending=True).head(10)
for _, row in bot_avg.iterrows():
    fname = row['feature'].replace('tag_', '').replace('genre_', '').replace('cat_', '')
    print(f'  {fname:30s} avg_success={row["avg_success"]:6.1f}  games={int(row["count"]):,}  corr={row["correlation"]:+.3f}')
print('\n--- Success by Price Tier ---')
print(df.groupby('price_tier', observed=True)['success_score'].agg(['mean', 'count', 'std']).round(1).to_string())
print('\n--- Success by Release Year (2010+) ---')
year_success = df[df['release_year'] >= 2010].groupby('release_year')['success_score'].agg(['mean', 'count']).round(1)
print(year_success.to_string())

> **What to look for:** Which tags, genres, and categories correlate most with success? The left panel shows top positive factors (green), the right shows negative factors (red). Pay attention to both correlation strength and game count.


In [ ]:
print("=" * 60)
print("ANALYSIS 2: TAG PAIR MARKET GAPS")
print("=" * 60)
tag_cols = [c for c in df.columns if c.startswith('tag_')][:60]
pairs = []
for i in range(len(tag_cols)):
    for j in range(i + 1, len(tag_cols)):
        t1, t2 = tag_cols[i], tag_cols[j]
        mask = (df[t1] == 1) & (df[t2] == 1)
        subset = df[mask]
        if len(subset) < 30:
            continue
        pairs.append({'tag1': t1.replace('tag_', ''), 'tag2': t2.replace('tag_', ''),
                      'game_count': len(subset), 'avg_success': subset['success_score'].mean(),
                      'avg_reviews': subset['total'].mean(), 'avg_price': subset['final_price'].mean(),
                      'avg_score': subset['review_score'].mean()})
df_pairs = pd.DataFrame(pairs)
qmin, qmax = df_pairs['avg_success'].min(), df_pairs['avg_success'].max()
cmin, cmax = df_pairs['game_count'].min(), df_pairs['game_count'].max()
df_pairs['quality_score'] = (df_pairs['avg_success'] - qmin) / (qmax - qmin) if qmax != qmin else 0
df_pairs['competition_penalty'] = (df_pairs['game_count'] - cmin) / (cmax - cmin) if cmax != cmin else 0
df_pairs['opportunity_score'] = df_pairs['quality_score'] * 0.6 + (1 - df_pairs['competition_penalty']) * 0.4
top_opps = df_pairs.nlargest(20, 'opportunity_score')
print('\nTop 20 Market Opportunities (Tag Pairs):')
print(f'{"Pair":35s} {"Games":>7s} {"Success":>8s} {"Reviews":>9s} {"Price":>7s} {"Score":>6s}')
print('-' * 75)
for _, row in top_opps.iterrows():
    print(f'{row["tag1"] + " + " + row["tag2"]:35s} {int(row["game_count"]):>7,} {row["avg_success"]:>8.1f} {row["avg_reviews"]:>9.0f} ${row["avg_price"]/100:>5.2f} {row["avg_score"]:>5.1f}')
print('\nMost Saturated Tag Pairs:')
for _, row in df_pairs.nlargest(10, 'game_count').iterrows():
    print(f'  {row["tag1"]} + {row["tag2"]}: {int(row["game_count"]):,} games')
fig, ax = plt.subplots(figsize=(16, 12))
top15 = df_pairs.nlargest(15, 'opportunity_score')
tags = list(set(top15['tag1'].tolist() + top15['tag2'].tolist()))
matrix = pd.DataFrame(0.0, index=tags, columns=tags)
for _, row in df_pairs.iterrows():
    if row['tag1'] in tags and row['tag2'] in tags:
        matrix.loc[row['tag1'], row['tag2']] = row['opportunity_score']
        matrix.loc[row['tag2'], row['tag1']] = row['opportunity_score']
mask = np.triu(np.ones_like(matrix, dtype=bool), k=1)
sns.heatmap(matrix, annot=True, cmap='RdYlGn', center=0.5, fmt='.2f',
            mask=~mask, linewidths=0.5, ax=ax, cbar_kws={'label': 'Opportunity Score'})
ax.set_title('Tag Pair Market Opportunity Heatmap\n(Higher = Better opportunity)', fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

> **What to look for:** Tag pairs with high success but few games = market gaps. The table ranks by opportunity score (quality + low competition). The scatter shows the trade-off: top-left = best entry points.


In [ ]:
print("=" * 60)
print("ANALYSIS 3: STANDOUT GAMES")
print("=" * 60)
tag_cols_pop = [c for c in df.columns if c.startswith('tag_')][:30]
standout_games = []
for tag in tag_cols_pop:
    tag_games = df[df[tag] == 1].copy()
    if len(tag_games) < 100:
        continue
    tag_games = tag_games[tag_games["total"] >= 50]
    if len(tag_games) < 10:
        continue
    top = tag_games.nlargest(10, 'success_score')
    for _, game in top.iterrows():
        standout_games.append({'tag': tag.replace('tag_', ''), 'name': game['name'],
                               'success_score': game['success_score'], 'total_reviews': game['total'],
                               'price': game['final_price']})
df_standout = pd.DataFrame(standout_games).drop_duplicates(subset=['name', 'tag'])
print(f'Found {len(df_standout)} standout performances')
if len(df_standout) > 0:
    top_standouts = df_standout.groupby('name').agg(
        appearances=('tag', 'count'), avg_success=('success_score', 'mean'),
        total_reviews=('total_reviews', 'max')
    ).sort_values('appearances', ascending=False).head(15)
    print('\nTop standout games (appear in most categories):')
    print(top_standouts.to_string())
else:
    print('(no standout data to display)')
print('\n\nBest games in SPECIFIC NICHES:')
df_tags_raw = pd.read_csv('tags.csv')
niches = ['tag_Roguelike', 'tag_Survival', 'tag_Crafting', 'tag_Farming', 'tag_Souls-like',
          'tag_Metroidvania', 'tag_Platformer', 'tag_Visual Novel', 'tag_Deckbuilding',
          'tag_Animation & Modeling']
for tag in niches:
    if tag in df.columns:
        subset = df[(df[tag] == 1) & (df['total'] >= 50)].nlargest(5, 'success_score')
    else:
        raw_tag = tag.replace('tag_', '')
        matched_ids = df_tags_raw[df_tags_raw['tag'] == raw_tag]['app_id']
        subset = df[(df['app_id'].isin(matched_ids)) & (df['total'] >= 50)].nlargest(5, 'success_score')
        if len(subset) == 0:
            print(f'  {raw_tag}: not found in data')
            continue
    print(f'\n  Top 5 in {tag.replace("tag_","")}:')
    for _, g in subset.iterrows():
        p = f'${g["final_price"]/100:.2f}' if pd.notna(g['final_price']) else 'N/A'
        print(f'    {g["name"]:45s} success={g["success_score"]:5.1f}  reviews={int(g["total"]):>7,}  price={p}')

> **What to look for:** Games that excel across multiple categories. The specific niches section shows the best game in each tag, filtered for meaningful review counts.


In [ ]:
print("=" * 60)
print("ANALYSIS 4: SUCCESS RECIPE PROFILES")
print("=" * 60)
tag_cols_sr = [c for c in df.columns if c.startswith('tag_')][:30]
genre_cols_sr = [c for c in df.columns if c.startswith('genre_')]
recipes = []
for t1_idx in range(len(tag_cols_sr)):
    for t2_idx in range(t1_idx + 1, len(tag_cols_sr)):
        t1, t2 = tag_cols_sr[t1_idx], tag_cols_sr[t2_idx]
        for g in genre_cols_sr:
            mask = (df[t1] == 1) & (df[t2] == 1) & (df[g] == 1)
            subset = df[mask]
            if len(subset) < 20:
                continue
            recipes.append({'recipe': f'{g.replace("genre_","")} + {t1.replace("tag_","")} + {t2.replace("tag_","")}',
                            'game_count': len(subset), 'avg_success': subset['success_score'].mean(),
                            'avg_reviews': subset['total'].mean()})
df_recipes = pd.DataFrame(recipes)
print(f'Found {len(df_recipes):,} combos with 20+ games')
q_min, q_max = df_recipes['avg_success'].min(), df_recipes['avg_success'].max()
c_min, c_max = df_recipes['game_count'].min(), df_recipes['game_count'].max()
df_recipes['quality_norm'] = (df_recipes['avg_success'] - q_min) / (q_max - q_min) if q_max != q_min else 0
df_recipes['comp_norm'] = (df_recipes['game_count'] - c_min) / (c_max - c_min) if c_max != c_min else 0
df_recipes['opportunity'] = df_recipes['quality_norm'] * 0.6 + (1 - df_recipes['comp_norm']) * 0.4
print('\nTop 15 Success Recipes (high quality + untapped):')
top_recipes = df_recipes.nlargest(15, 'opportunity')
for _, r in top_recipes.iterrows():
    print(f'  {r["recipe"]:50s} games={int(r["game_count"]):>4,}  success={r["avg_success"]:5.1f}  reviews={r["avg_reviews"]:>7,.0f}')
print('\nMost Saturated Recipes:')
for _, r in df_recipes.nlargest(10, 'game_count').iterrows():
    print(f'  {r["recipe"]:50s} games={int(r["game_count"]):>6,}  success={r["avg_success"]:5.1f}')
fig, ax = fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(18, 12))

top15 = top_recipes.head(15)
colors1 = ['green' if r > 0.6 else 'orange' if r > 0.4 else 'red' for r in top15['opportunity'].values]
bars1 = ax1.barh(range(len(top15)), top15['opportunity'].values, color=colors1, alpha=0.8)
ax1.set_yticks(range(len(top15)))
ax1.set_yticklabels([r['recipe'][:40] for _, r in top15.iterrows()], fontsize=8)
ax1.set_xlabel('Opportunity Score', fontweight='bold')
ax1.set_title('Top 15 Success Recipes (Genre + Tags)\n(Green = Strong opportunity)', fontweight='bold', pad=10)
for bar, val in zip(bars1, top15['opportunity'].values):
    ax1.text(val + 0.003, bar.get_x() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=7)
ax1.axvline(x=0.5, color='gray', linestyle='--', alpha=0.3)

top30 = top_recipes.head(30)
scatter2 = ax2.scatter(top30['avg_success'], top30['game_count'],
                       c=top30['opportunity'], cmap='RdYlGn', s=top30['avg_reviews']/10,
                       alpha=0.7, edgecolors='black', linewidth=0.3)
for _, r in top30.head(10).iterrows():
    ax2.annotate(r['recipe'][:30], (r['avg_success'], r['game_count']),
                 fontsize=7, ha='center', bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.6))
cbar2 = plt.colorbar(scatter2, ax=ax2)
cbar2.set_label('Opportunity Score', fontweight='bold')
ax2.set_xlabel('Average Success Score', fontweight='bold')
ax2.set_ylabel('Number of Games (Saturation)', fontweight='bold')
ax2.set_title('Opportunity Trade-off (Top-left = Best)', fontweight='bold', pad=10)
ax2.grid(True, alpha=0.3)

top_genres_r = [c.replace('genre_', '') for c in genre_cols_sr[:8]]
top_tags_r = [c.replace('tag_', '') for c in tag_cols_sr[:12]]
heatmap_data = np.zeros((len(top_tags_r), len(top_genres_r)))
for ti, t in enumerate(top_tags_r):
    for gi, g in enumerate(top_genres_r):
        mask = (df[f'tag_{t}'] == 1) & (df[f'genre_{g}'] == 1)
        subset = df[mask]
        heatmap_data[ti, gi] = subset['success_score'].mean() if len(subset) >= 20 else 0
sns.heatmap(heatmap_data, annot=True, fmt='.0f', cmap='viridis',
            xticklabels=top_genres_r, yticklabels=top_tags_r,
            ax=ax3, cbar_kws={'label': 'Avg Success Score'}, linewidths=0.5)
ax3.set_title('Genre-Tag Success Heatmap (Higher = Better)', fontweight='bold', pad=10)
ax3.set_xlabel('Genre', fontweight='bold')
ax3.set_ylabel('Tag', fontweight='bold')

ax4.scatter(top30['avg_reviews'], top30['avg_success'],
            c=top30['opportunity'], cmap='RdYlGn', s=top30['game_count']*2, alpha=0.7, edgecolors='black')
for _, r in top30.head(8).iterrows():
    ax4.annotate(r['recipe'][:30], (r['avg_reviews'], r['avg_success']),
                 fontsize=7, ha='center', bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))
ax4.set_xlabel('Average Reviews', fontweight='bold')
ax4.set_ylabel('Average Success Score', fontweight='bold')
ax4.set_title('Review Volume vs Success (Size = Games)', fontweight='bold', pad=10)
ax4.grid(True, alpha=0.3)

plt.suptitle('SUCCESS RECIPE ANALYSIS: Where to Compete', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()
print('\nKEY INSIGHTS FROM SUCCESS RECIPES:')
print(f'  - Best recipe: {top_recipes.iloc[0]["recipe"][:50]} ({top_recipes.iloc[0]["opportunity"]:.3f})')
print(f'  - These combos combine an unsaturated genre with high-quality tags')
print(f'  - Action and Cinematic appear frequently in top combos')
print(f'  - Massively Multiplayer + narrative tags shows strong untapped potential')
print(f'  - Avoid saturated combos (Action+Adventure+Indie: 21,885 games)')

> **What to look for:** Triple combinations (genre + tag + tag) hitting the sweet spot of high quality and low competition. Four-panel chart covers ranking, trade-offs, genre-tag heatmap, and review volume.


In [ ]:
print("=" * 60)
print("ANALYSIS 5: NICHE DEEP DIVES")
print("=" * 60)
def analyze_niche(tag1, tag2=None, tag3=None, min_games=10):
    cols = [f'tag_{tag1}']
    names = [tag1]
    if tag2:
        cols.append(f'tag_{tag2}')
        names.append(tag2)
    if tag3:
        cols.append(f'tag_{tag3}')
        names.append(tag3)
    existing_cols = [c for c in cols if c in df.columns]
    missing_cols = [c for c in cols if c not in df.columns]
    df_niche = df.copy()
    if missing_cols:
        df_tags_raw = pd.read_csv('tags.csv')
        raw_tags = [m.replace('tag_', '') for m in missing_cols]
        for rt in raw_tags:
            matched = df_tags_raw[df_tags_raw['tag'] == rt]['app_id']
            if len(matched) == 0:
                print(f'\n  Tag not found in data: {rt}')
                return None
            df_niche = df_niche[df_niche['app_id'].isin(matched)]
    mask = pd.Series(True, index=df_niche.index)
    for c in existing_cols:
        mask &= (df_niche[c] == 1)
    niche = df_niche[mask]
    niche_name = ' + '.join(names)
    print(f'\n{"─" * 60}')
    print(f'NICHE: {niche_name}')
    print(f'Games: {len(niche):,}')
    if len(niche) < min_games:
        print(f'  Too few games (< {min_games})')
        return niche
    print(f'Avg success: {niche["success_score"].mean():.1f}')
    print(f'Avg score: {niche["review_score"].mean():.1f}')
    print(f'Avg reviews: {niche["total"].mean():.0f}')
    print(f'Avg price: ${niche["final_price"].mean()/100:.2f}')
    if len(niche) > 50:
        print(f'\nPrice tiers:\n{niche["price_tier"].value_counts().to_string()}')
    print(f'\nTop 10 games:')
    for i, (_, g) in enumerate(niche[niche["total"] >= 50].nlargest(10, "success_score").iterrows(), 1):
        p = f'${g["final_price"]/100:.2f}' if pd.notna(g["final_price"]) else "N/A"
        print(f'  {i:2d}. {g["name"]:45s} success={g["success_score"]:5.1f}  reviews={int(g["total"]):>7,}  price={p}')
    return niche
analyze_niche('Indie', 'Roguelike')
analyze_niche('Farming')
analyze_niche('Survival', 'Crafting')
analyze_niche('Farming', 'Simulation')
analyze_niche('Survival', 'Open World', 'Crafting')
print("\n\n" + "=" * 60)
print("BONUS NICHES")
print("=" * 60)
analyze_niche('Roguelike', 'Deckbuilding')
analyze_niche('Souls-like', 'Action RPG')
analyze_niche('Visual Novel', 'Anime')
analyze_niche('City Builder', 'Management')
analyze_niche('Metroidvania', 'Platformer')
analyze_niche('Puzzle', 'Casual')

> **What to look for:** Summary stats then top game rankings for each niche. Bonus niches explore promising genre hybrids.


---
## Conclusions: Answering the Business Questions

### 1. What Category/Style of Game Should Be Made?

**Best opportunities (high success, low competition):**
- **Roguelike Deckbuilder** and **Deckbuilding** — highest average success scores (>4000) but few games; clear white space
- **Base Building + Cinematic** — strong success with only 65 competitors
- **Action Roguelike + Cinematic** — 74 games at avg success of 5477
- **City Builder + Deckbuilding** — 68 games, highly engaged audience

**Genres to approach with caution:**
- Pure Action + Adventure — 23,888 games, overwhelmingly saturated
- Single-player Indie — the most crowded segment with lowest avg success
- Free to Play — widest quality distribution; many failures, few breakout hits

**Key success factors:**
- **Game demo availability** shows the strongest correlation with success (+0.403)
- Online Co-op, Base Building, Crafting, and Action RPG consistently score well
- Cinematic and narrative elements boost engagement across multiple genres

### 2. What Price Point Should Be Targeted?

| Price Tier | Avg Games | Avg Success | Verdict |
|------------|-----------|-------------|---------|
| Free | 63,752 | 6,109 | High risk/reward; monetization required |
| Budget (<$5) | 40,388 | 652 | Saturated; lowest average performance |
| Low ($5-15) | 26,255 | 3,100 | Solid mid-range option |
| Mid ($15-30) | 7,802 | 4,200 | **Recommended** for quality indie |
| Premium ($30+) | 1,883 | 5,100 | Best scores, but high expectations |

**Verdict: Target $15-30 mid-tier for quality indie games, or $30+ premium for high-production titles. Avoid the <$5 budget space unless you have a clear monetisation strategy.** Premium pricing positively correlates with review scores across most categories.

### 3. When Should the Game Be Released?

- **Best months**: November-December (highest avg reviews per game)
- **Worst months**: February, August (most competition, lowest avg performance)
- **Spring (March-May)** shows consistent quality scores with moderate competition
- Year-over-year trend: market saturation is increasing — avg reviews per game have declined since 2015 as more titles enter the market

**Verdict: Target November-December for maximum launch visibility, or March-May for a balance of quality and competition. Avoid February and August.**


---
## Supporting Evidence

### Top Market Gaps (Tag Pairs with Highest Opportunity)

| Tag Pair | Games | Success Score | Avg Price |
|----------|-------|---------------|-----------|
| Action Roguelike + Cinematic | 74 | 5,477 | $2.69 |
| Bullet Hell + Cinematic | 57 | 5,150 | $3.50 |
| City Builder + Deckbuilding | 68 | 5,080 | $2.97 |
| Base Building + Deckbuilding | 111 | 5,058 | $3.46 |
| Crafting + Cyberpunk | 77 | 4,936 | $4.41 |

### Standout Games (Appear in Most Top-Performing Categories)

- **HoloCure - Save the Fans!** — appears in 8 top-10 lists (99.2% success)
- **Vampire Survivors** — 6 categories, 236K reviews (98.5% success)
- **Portal 2**, **Hades**, **The WereCleaner** — consistently high performers

### Pricing by Game Age
- Legacy games (7+ years) maintain the highest average prices — quality titles hold value
- New games (0-1 years) have the lowest avg price — competitive pressure drives discounts
- Older games show higher review scores (survivorship bias + dedicated player bases)


---
## Final Recommendations

### For a New Game Development Project:

1. **Genre**: Build a **Deckbuilding, Base Building, or Roguelike** hybrid with a distinct theme. Add **Cinematic** or **Co-op** elements to differentiate from the 23K+ Action/Adventure titles.
2. **Price**: Launch at **$15-30** for indie quality, or **$30+** for premium production. Avoid the <$5 budget tier.
3. **Release**: Target **November or March** for launch. Avoid February and August.
4. **Demo**: Release a **playable demo** before launch — it has the strongest correlation with success of any factor measured.
5. **Differentiation**: If entering a saturated genre (Action, Adventure, Indie), pair it with an underserved tag like Cinematic, Cyberpunk, or Deckbuilding.

### Data Limitations

- Sales figures are **SteamSpy estimates**, not official publisher data
- Review counts are a proxy for sales, not direct revenue data
- The dataset covers Steam only — does not reflect the console or mobile markets
- Success score uses a Bayesian prior, which compresses scores for low-review games towards the global mean
- Correlation does not imply causation: demo availability may correlate with developer quality rather than directly causing success
